In [ ]:
import os
from pathlib import Path

# Set GCP credentials dynamically
cred_path = Path.cwd().parent / "dev" / "credentials.json"
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = str(cred_path.resolve())

import pandas as pd

import plotly.express as px
from plotly.graph_objects import Figure
from dashboard.data import (
    get_trips_by_day_type,
    get_trips_by_hour,
    get_trips_by_month,
    get_station_activity,
    get_avg_duration_by_customer,
)

In [ ]:
# table = _make_mart_table_name("mart_trips_by_day_type")
# day_type, customer_type, rideable_type, trip_count

df_day = get_trips_by_day_type()


def chart_trips_by_day_type(df: pd.DataFrame) -> Figure:
    """Chart showing trips by day type (weekday, weekend).

    Chart type: bar chart
    Axis x: day_type
    Axis y: trip_count
    """

    labels = {
        "day_type": "Day Type",
        "trip_count": "Total Trips",
        "customer_type": "Customer Type",
    }
    colors = {
        "member": "#1f77b4",  # blue
        "casual": "#ff7f0e",  # orange
    }
    fig = px.bar(
        df,
        title="Day Type Patterns",
        x="day_type",
        y="trip_count",
        labels=labels,
        color="customer_type",
        barmode="group",
        color_discrete_map=colors,
    )
    return fig


chart_trips_by_day_type(df_day).show()

In [ ]:
# trip_date, trip_hour, customer_type, rideable_type, trip_count


df_hour = get_trips_by_hour()


def chart_trips_by_hour(df: pd.DataFrame) -> Figure:
    """Chart showing trips by hour.

    Chart type: bar chart
    Axis x: trip_hour
    Axis y: trip_count
    """
    df = (
        df.filter(["trip_hour", "trip_count", "customer_type"])
        .groupby(["trip_hour", "customer_type"], as_index=False)
        .sum()
    )

    labels = {
        "trip_hour": "Day Hour",
        "trip_count": "Total Trips",
        "customer_type": "Customer Type",
    }
    colors = {
        "member": "#1f77b4",  # blue
        "casual": "#ff7f0e",  # orange
    }

    fig = px.bar(
        df,
        title="Peak Hours",
        x="trip_hour",
        y="trip_count",
        labels=labels,
        color="customer_type",
        barmode="stack",
        color_discrete_map=colors,
    )

    # Force x-axis to show every hour tick label
    fig.update_layout(
        xaxis=dict(
            tickmode="linear",
            tick0=0,
            dtick=1,
        )
    )

    return fig


chart_trips_by_hour(df_hour).show()

In [ ]:
# year, month, customer_type, rideable_type, trip_count

df_month = get_trips_by_month()


def chart_trips_by_month(df: pd.DataFrame) -> Figure:
    """Chart showing trips by month.

    Chart type: line chart
    Axis x: year-month
    Axis y: trip_count
    """
    df = (
        df.filter(["year", "month", "trip_count", "customer_type"])
        .groupby(["year", "month", "customer_type"], as_index=False)
        .sum()
    )
    df["year_month"] = (
        df["year"].astype(str) + "-" + df["month"].astype(str).str.zfill(2)
    )

    labels = {
        "trip_count": "Total Trips",
        "customer_type": "Customer Type",
        "year_month": "Month",
    }
    colors = {
        "member": "#1f77b4",  # blue
        "casual": "#ff7f0e",  # orange
    }

    fig = px.line(
        df,
        title="Monthly Trends",
        x="year_month",
        y="trip_count",
        labels=labels,
        color="customer_type",
        color_discrete_map=colors,
    )

    # Format x-axis: monthly ticks + vertical labels
    fig.update_layout(
        xaxis=dict(
            dtick="M1",  # one month interval
            tickformat="%Y-%m",  # show Year-Month
            tickangle=90,  # vertical labels
        )
    )

    return fig


chart_trips_by_month(df_month).show()

In [ ]:
# station_name, station_role, trip_count

df_station_start = get_station_activity()


def chart_station_activity_start(df: pd.DataFrame) -> Figure:
    """Chart showing top stations.

    Chart type: horizontal bar chart
    Axis x: trip_count
    Axis y: station_name
    """
    df = df.where(df["station_role"] == "start").sort_values("trip_count")

    labels = {
        "trip_count": "Total Trips",
        "station_name": "Station Name",
        "station_role": "Station Role",
    }
    colors = {
        "start": "#1f77b4",  # blue
    }

    fig = px.bar(
        df,
        title="Top Start Stations",
        x="trip_count",
        y="station_name",
        labels=labels,
        color="station_role",
        color_discrete_map=colors,
        orientation="h",
    )

    # Format y-axis
    fig.update_layout(
        showlegend=False,
        yaxis=dict(
            automargin=True,
            title_text="",
        ),
    )

    return fig


chart_station_activity_start(df_station_start).show()

In [ ]:
# station_name, station_role, trip_count

df_station_end = get_station_activity()


def chart_station_activity_end(df: pd.DataFrame) -> Figure:
    """Chart showing top stations.

    Chart type: horizontal bar chart
    Axis x: trip_count
    Axis y: station_name
    """
    df = df.where(df["station_role"] == "end").sort_values("trip_count")

    labels = {
        "trip_count": "Total Trips",
        "station_name": "Station Name",
        "station_role": "Station Role",
    }
    colors = {
        "end": "#ff7f0e",  # orange
    }

    fig = px.bar(
        df,
        title="Top End Stations",
        x="trip_count",
        y="station_name",
        labels=labels,
        color="station_role",
        color_discrete_map=colors,
        orientation="h",
    )

    # Format y-axis
    fig.update_layout(
        showlegend=False,
        yaxis=dict(
            automargin=True,
            title_text="",
        ),
    )

    return fig


chart_station_activity_end(df_station_end).show()

In [ ]:
# customer_type, rideable_type, avg_trip_duration_minutes, median_trip_duration_minutes

df_duration = get_avg_duration_by_customer()


def chart_avg_duration_by_customer(df: pd.DataFrame) -> Figure:
    """Chart showing average trip duration by customer.

    Chart type: grouped bar chart
    Axis x: customer_type + rideable_type
    Axis y: avg_trip_duration
    """
    df = df.melt(
        id_vars=["customer_type", "rideable_type"],
        value_vars=["avg_trip_duration_minutes", "median_trip_duration_minutes"],
        var_name="metric",
        value_name="value",
    )

    labels = {
        # "avg_trip_duration_minutes": "Average Trip Duration",
        # "median_trip_duration_minutes": "Median Trip Duration",
        "metric": "Metric",
        "value": "Minutes",
        "customer_type": "Customer Type",
        "rideable_type": "Rideable Type",
    }
    colors = {
        "avg_trip_duration_minutes": "#1f77b4",  # blue
        "median_trip_duration_minutes": "#ff7f0e",  # orange
    }

    fig = px.bar(
        df,
        title="Average Trip Duration (minutes)",
        x="customer_type",
        y="value",
        labels=labels,
        barmode="group",
        facet_col="rideable_type",
        color="metric",
        color_discrete_map=colors,
    )

    # Format layout
    fig.update_layout(
        yaxis=dict(
            title="Minutes",
        )
    )

    return fig


chart_avg_duration_by_customer(df_duration).show()